# 01 — Exploratory Data Analysis: Dogs vs Cats

**Goal**: get a feel for the dataset before training. Class balance, image dimensions, channel statistics, and a visual sample grid.

All heavy lifting is delegated to the `src/` package. This notebook is the *story*, not the implementation.

In [ ]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH so we can import src.*
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from collections import Counter

from src.data import download_dataset
from src.utils import set_seed

set_seed(42)

## 1. Locate the dataset

`download_dataset` pulls from Kaggle Hub if needed and finds the `ImageFolder` root.

In [ ]:
data_root = download_dataset()
print(f'Dataset root: {data_root}')
classes = sorted([d.name for d in data_root.iterdir() if d.is_dir()])
print(f'Classes:      {classes}')

## 2. Class balance

If one class dominates, we'll need to address it (resampling, weighted loss). Let's check first.

In [ ]:
counts = {cls: len(list((data_root / cls).iterdir())) for cls in classes}
total  = sum(counts.values())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.keys(), counts.values(), color=['#f97316', '#3b82f6'])
for b, c in zip(bars, counts.values()):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + total*0.005,
            f'{c} ({c/total*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Image count per class')
ax.set_ylabel('Number of images')
plt.tight_layout(); plt.show()

print(f'\nTotal: {total} images | Imbalance ratio: {max(counts.values())/min(counts.values()):.2f}')

**Reading**: a ratio near 1.0 means the dataset is balanced — we can safely use plain cross-entropy without class weights.

## 3. Image dimension distribution

Are images all the same size? If not, what's the dynamic range? This affects the resize strategy.

In [ ]:
widths, heights = [], []
sample_paths = []
for cls in classes:
    for p in list((data_root / cls).iterdir())[:300]:  # sample 300/class for speed
        try:
            with Image.open(p) as im:
                widths.append(im.size[0]); heights.append(im.size[1])
        except Exception:
            continue
        sample_paths.append((p, cls))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=40, color='#3b82f6', alpha=0.85)
axes[0].set_title('Width distribution'); axes[0].set_xlabel('pixels')
axes[1].hist(heights, bins=40, color='#f97316', alpha=0.85)
axes[1].set_title('Height distribution'); axes[1].set_xlabel('pixels')
plt.tight_layout(); plt.show()

print(f'Width  — min {min(widths)}, median {int(np.median(widths))}, max {max(widths)}')
print(f'Height — min {min(heights)}, median {int(np.median(heights))}, max {max(heights)}')

**Reading**: images vary substantially in size. Our pipeline resizes everything to 64×64 — a deliberate compromise between training speed and information retained. A 224×224 resize is the obvious next experiment.

## 4. Sample grid

Always look at your data with your own eyes.

In [ ]:
import random

rng = random.Random(42)
n_per = 4
fig, axes = plt.subplots(2, n_per, figsize=(3*n_per, 6))
for row, cls in enumerate(classes):
    cls_paths = list((data_root / cls).iterdir())
    rng.shuffle(cls_paths)
    for col in range(n_per):
        img = Image.open(cls_paths[col]).convert('RGB')
        axes[row, col].imshow(img); axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=14, fontweight='bold')
fig.suptitle('Random samples from each class', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Takeaways for training

- ✅ Classes are roughly balanced — plain cross-entropy is fine.
- ✅ Image sizes are varied — we resize to 64×64.
- ✅ Visual variability is high (pose, lighting, framing) — augmentation should help.
- 🎯 Next steps: train the baseline ([notebook 02](02_baseline_logreg.ipynb)) and the CNN ([notebook 03](03_cnn_training.ipynb)).